# Original pretrained-model protocol: set A
This notebook reproduces the original short-context protocol. For the final repaired long-context evaluation, use the notebooks in `dram/`. Choose a CUDA GPU runtime and run the cells in order. The first code cell clones the repository and restores prepared IR. The checkpoint is pinned SmolLM2-135M-Instruct, FP32. SetA is the no-match control; SetB tests60 concat-commit rewrites. Numerical failures remain recorded.


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import files
repo = Path('/content/cs6886_project')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/lazarsalt1441/cs6886_project.git', str(repo)], check=True)
subprocess.run([sys.executable, str(repo / 'scripts/unpack_artifacts.py')], check=True)
project = repo / 'real_model'
print('Project:', project)


In [ ]:
import os, subprocess, sys
venv = Path("/content/attention_real_env")
# Bundled pip avoids the previous ensurepip failure. Retain Colab CUDA Torch.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", str(venv)], check=True)
python = str(venv / "bin/python")
env = os.environ.copy()
env["PATH"] = str(venv / "bin") + os.pathsep + env["PATH"]
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
with (project / "setup.log").open("w") as log:
    result = subprocess.run([python, "-m", "pip", "install", "-r", str(project / "requirements-colab.txt")], env=env, stdout=log, stderr=subprocess.STDOUT)
if result.returncode:
    print((project / "setup.log").read_text()[-8000:])
    raise RuntimeError("Setup failed; upload setup.log")
subprocess.run([python, "-c", "import text_only; import torch, iree.runtime as r; from transformers import LlamaForCausalLM; print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda); assert torch.cuda.is_available(), 'Choose GPU runtime'; print('GPU:', torch.cuda.get_device_name()); print('IREE:', r.get_driver('cuda').query_available_devices())"], cwd=project, env=env, check=True)
print("Setup PASS")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
backup_root = Path("/content/drive/MyDrive/Attention_Real_LLM_Results")
backup_root.mkdir(parents=True, exist_ok=True)
print("Drive backup ready")


Default: contexts 128/512/2048, two prompts, 31 trials × 3 repetitions, plus 16-token continuations. Torch compilation is outside measured trials and may take several minutes per shape. Set `QUICK=True` for a first pipeline check; use `False` for final measurements. The original protocol records reference backends; the submitted presentation uses the IREE comparisons. Unsupported Torch compilation is logged and marked unavailable.

Leave the next cell running. It prints a heartbeat every minute and saves a partial Drive archive every five minutes. This does not keep a disconnected Colab runtime alive.


In [ ]:
SET = "A"
QUICK = False
SKIP_TORCH_COMPILE = True

import signal, shutil, time
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
backup_dir = backup_root / ("set_" + SET + "_" + stamp)
backup_dir.mkdir(parents=True, exist_ok=True)
archive = project / ("real_llm_set_" + SET + "_" + stamp + ".tar.gz")
log_path = project / ("runner_" + SET + ".log")

def save_snapshot():
    subprocess.run([python, "scripts/archive_results.py", "--set", SET, "--destination", str(archive)], cwd=project, env=env, check=True, stdout=subprocess.DEVNULL)
    shutil.copy2(archive, backup_dir / archive.name)
    if log_path.exists(): shutil.copy2(log_path, backup_dir / log_path.name)
    if (project / "setup.log").exists(): shutil.copy2(project / "setup.log", backup_dir / "setup.log")

cmd = [python, "-u", "scripts/run_colab_set.py", "--set", SET]
if QUICK: cmd.append("--quick")
if SKIP_TORCH_COMPILE: cmd.append("--skip-torch-compile")
proc = None
try:
    with log_path.open("w", buffering=1) as log:
        proc = subprocess.Popen(cmd, cwd=project, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        start = last_backup = time.monotonic()
        while proc.poll() is None:
            try: proc.wait(timeout=60)
            except subprocess.TimeoutExpired: pass
            print("Elapsed minutes:", round((time.monotonic()-start)/60, 1), flush=True)
            print("\n".join(log_path.read_text(errors="replace").splitlines()[-3:]), flush=True)
            if time.monotonic()-last_backup >= 300:
                save_snapshot(); last_backup=time.monotonic()
                print("Partial backup saved", flush=True)
        print("Exit code:", proc.returncode)
        if proc.returncode: print(log_path.read_text(errors="replace")[-10000:])
except BaseException:
    if proc is not None and proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try: proc.wait(timeout=10)
        except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL)
    raise
finally:
    save_snapshot()
    print("Saved to Drive:", backup_dir, flush=True)
    print("Starting browser download:", archive.name, flush=True)
    files.download(str(archive))


The resulting `real_llm_set_*.tar.gz` archives preserve the complete experiment records. They include correctness/errors, randomized timings, compilation logs, generation tokens, environment, MLIR and PTX when available. Model weights are excluded. If browser download is blocked, use Drive or rerun the next cell.


In [ ]:
# Run only to repeat the download.
files.download(str(archive))
